In [11]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
# H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
mol.precision = 1e-10
mol.spin = 3
mol.build()
mf = scf.UHF(mol).density_fit()
mf = mf.newton()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-23-generic', version='#23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Tue May 12 14:35:20 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 5
[INPUT] num. electrons = 19
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 3
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [ ]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}') 

MP2 Corr = -0.31608202
CCSD Corr = -0.34355579
CCSD(T) Corr = -0.34834121


In [20]:
from pyscf.lno import ULNOCCSD
from pyscf.lno.tools import autofrag_iao

# def test_lno_iao_by_thresh(self):
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc_a = mf.mo_coeff[0][:,frozen:np.count_nonzero(mf.mo_occ[0])]
orbocc_b = mf.mo_coeff[1][:,frozen:np.count_nonzero(mf.mo_occ[1])]
lo_coeff_a = lo.iao.iao(mol, orbocc_a)
lo_coeff_b = lo.iao.iao(mol, orbocc_b)
lo_coeff_a = lo.orth.vec_lowdin(lo_coeff_a, mf.get_ovlp())
lo_coeff_b = lo.orth.vec_lowdin(lo_coeff_b, mf.get_ovlp())
lo_coeff = [lo_coeff_a, lo_coeff_b]
moliao = lo.iao.reference_mol(mol)

frag_lolist = autofrag_iao(moliao)
frag_lolist = [[frag,frag] for frag in frag_lolist]
print(frag_lolist)

[[[0, 1, 2, 3, 4], [0, 1, 2, 3, 4]], [[5], [5]], [[6], [6]], [[7, 8, 9, 10, 11], [7, 8, 9, 10, 11]], [[12], [12]]]


In [21]:
print(lo_coeff_a.shape)
print(lo_coeff_b.shape)

(43, 13)
(43, 13)


In [22]:
# from collections.abc import Iterable
from pyscf.lno import lnoccsd
from pyscf.lno import ulnoccsd
from afqmc.lno_afqmc.mod_lnoccsd import lnoccsd_kernel
from afqmc.lno_afqmc import prep
import jax
from jax import random
import os
from collections.abc import Iterable
import time
import gc

In [25]:
def run_afqmc(mf,
              lo_coeff = None, 
              lo_coeff_file = 'lo_coeff.npz',
              frag_lolist = None,
              nfrozen = 0,
              thresh = 1e-6,
              qmc_options = {}, 
              chol_cut = 1e-5, 
              target_sto_error = 1e-3, 
              run_frg_list = None, 
              atom_group = None,
              ):
    
    if lo_coeff is None:
        try:
            lo_coeff = np.load(lo_coeff_file)["lo_coeff"]
        except:
            raise ValueError(
                f"lo_coeff was not provided and could not be loaded from '{lo_coeff_file}'")

    spin_type = prep.kind(lo_coeff)

    if frag_lolist is None:
        if spin_type == "unrestricted":
            raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
        print("Fragment list not found. Asign every LO to a fragment.")
        frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

    if spin_type == "unrestricted":
        mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
        mf = mlno._scf
    else:
        mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mlno.lno_thresh = [thresh*10, thresh]
    lno_thresh = mlno.lno_thresh
    lno_type = ['1h','1h']
    eris = mlno.ao2mo()

    nfrag = len(frag_lolist)
    if run_frg_list is None:
        run_frg_list = range(nfrag)
    
    frag_lolist = [frag_lolist[i] for i in run_frg_list]
    lno_pct_occ = [None, None]
    lno_norb = [[None,None]] * nfrag

    # seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
    #                        shape=(nfrag,), 
    #                        minval=0, 
    #                        maxval=100*nfrag
    #                        )
    
    # qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag)
    # trial_base = qmc_options.get("trial", "")

    las_center = [None]*nfrag
    las_size = np.zeros(nfrag, dtype='int32')
    lno_emp2 = np.zeros(nfrag, dtype='float64')
    lno_ecc  = np.zeros(nfrag, dtype='float64')
    lno_eqmc = np.zeros(nfrag, dtype='float64')
    lno_eqmc_err  = np.zeros(nfrag, dtype='float64')
    ccsd_time = np.zeros(nfrag, dtype='float64')
    qmc_time = np.zeros(nfrag, dtype='float64')

    # Loop over fragment
    for ifrag, loidx in enumerate(frag_lolist):
        print("\n")
        width = 80
        msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
        print(msg.center(width, '='))
        if atom_group is not None:
            atom_msg = f"{atom_group[ifrag]}"
            print(f"Center Atom {atom_msg}")

        if spin_type == "unrestricted":
            orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
            lno_param = [
                [
                    {
                        'thresh': (
                            lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                            else lno_thresh[i]
                        ),
                        'pct_occ': (
                            lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                            else lno_pct_occ[i]
                        ),
                        'norb': (
                            lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                            else lno_norb[ifrag][i]
                        ),
                    } for i in [0, 1]
                ] for s in range(2)
            ]
        else:
            orbloc = lo_coeff[:,loidx]
            lno_param = [{
                'thresh': lno_thresh[i],
                'pct_occ': lno_pct_occ[i],
                'norb': lno_norb[ifrag][i]
                } for i in [0,1]]

        # M = <orbloc|canactocc> (M^dagger M)u = eu
        # u|canactocc> => orbtial in/out the space spanned by |orbloc>
        # uocc_loc = <lno_actocc|orbloc>
        lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)
        # lno_coeff still connected to canonical mo_coeff unitarily

        if spin_type == "unrestricted":
            if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
                lno_elec_type = 'alpha'
                # spin_idx = 0
            elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
                lno_elec_type = 'beta'
                # spin_idx = 1
            else:
                lno_elec_type = 'mixed'
                # spin_idx = 0
            print(f'LNO-Frgament Spin Type = {lno_elec_type}')
        #     ao_message, ao_max = prep.ao_comp(mf, orbloc[spin_idx]) # TODO: make it general
        # else:
        #     ao_message, ao_max = prep.ao_comp(mf, orbloc)

        mo_occ = mlno.mo_occ

        if spin_type == "unrestricted":
            lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
            occidxa = mo_occ[0] > 1e-10
            occidxb = mo_occ[1] > 1e-10
            moidxa, moidxb = maskact
            nactocc_a = int(np.sum(moidxa & occidxa))
            nactvir_a = int(np.sum(moidxa & ~occidxa))
            nactocc_b = int(np.sum(moidxb & occidxb))
            nactvir_b = int(np.sum(moidxb & ~occidxb))
            nactocc = nactocc_a + nactocc_b
            nactvir = nactvir_a + nactvir_b
            print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
            print(f'LAS beta:  {nactocc_b} occupied, {nactvir_b} virtual')
        else:
            lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
            nactocc, nactvir = prep.las_size(mf, lno_frozen)
            print(f'LAS occupied orbitals: {nactocc}')
            print(f'LAS virtual orbitals: {nactvir}')

        if spin_type == "unrestricted":
            mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
        else:
            mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
        mcc._s1e = mlno._s1e
        mcc._h1e = mlno._h1e
        mcc._vhf = mlno._vhf
        if mlno.kwargs_imp is not None:
            mcc = mcc.set(**mlno.kwargs_imp)
        time0 = time.perf_counter()
        (eorb_mp2, eorb_cc), t1, t2 =\
            lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
        time1 = time.perf_counter()
        lnocc_time = time1 - time0

        print(f"CCSD time: {lnocc_time:.6f} s")
        print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
        print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

        if atom_group:
            las_center[ifrag] = atom_msg
        # else:
        #     las_center[ifrag] = ao_max
        las_size[ifrag] = nactocc + nactvir
        lno_emp2[ifrag] = eorb_mp2
        lno_ecc[ifrag] = eorb_cc
        ccsd_time[ifrag] = lnocc_time

        # # project onto center lo space
        # # <lno_actocc|orbloc> <orbloc|lno_actocc>
        # if spin_type == "unrestricted":
        #     prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
        #              uocc_loc[1] @ uocc_loc[1].T.conj()]
        #     qmc_options["trial"] = trial_base
        #     if 'ad' not in trial_base:
        #         if lno_elec_type == 'alpha':
        #             qmc_options["trial"] += '_alpha'
        #         elif lno_elec_type == 'beta':
        #             qmc_options["trial"] += '_beta'
        # else:
        #     prjlo = uocc_loc @ uocc_loc.T.conj()

        # qmc_options["seed"] = seeds[ifrag]
        # prep.prep_afqmc_integral(
        #     mf,
        #     lno_coeff,
        #     t1,
        #     t2,
        #     lno_frozen,
        #     prjlo,
        #     qmc_options,
        #     chol_cut=chol_cut
        #     )
        
        # run_lnoafqmc(qmc_options)
        # outfile = f'fragment.out{run_frg_list[ifrag]+1}'
        # os.system(f'mv afqmc.out {outfile}')
        # with open(outfile, "r") as f:
        #     for line in f:
        #         if "Blocked AFQMC/pt2CCSD Orbital Energy" in line:
        #             eorb_afqmc = float(line.split()[-3])
        #             eorb_afqmc_err = float(line.split()[-1])
        #         if "total run time" in line:
        #             lnoqmc_time = float(line.split()[-1])
        # lno_eqmc[ifrag] = eorb_afqmc
        # lno_eqmc_err[ifrag] = eorb_afqmc_err
        # qmc_time[ifrag] = lnoqmc_time

        # header = f' Fragment{run_frg_list[ifrag]+1} Results '
        # width = 80  # pick a consistent total width
        # with open(outfile, 'a') as f:
        #     f.write('\n')
        #     f.write(f'{header:=^{width}}\n')
        #     if atom_group:
        #         f.write("\t Center Atom " + atom_msg + "\n")
        #     f.write("\t" + ao_message + "\n")
        #     f.write('-' * width + '\n')
        #     f.write(f'\t LNO-Active Space electrons: {nactocc} | orbitals: {nactocc+nactvir} \n')
        #     f.write(f'\t LNO-MP2 Orbital Energy:   {eorb_mp2:.8f} \n')
        #     f.write(f'\t LNO-CCSD Orbital Energy:  {eorb_cc:.8f} \n')
        #     f.write(f'\t LNO-AFQMC Orbital Energy: {eorb_afqmc:.6f} +/- {eorb_afqmc_err:.6f} \n')
        #     f.write(f'\t LNO-CCSD Time:  {lnocc_time:.2f} \n')
        #     f.write(f'\t LNO-AFQMC Time: {lnoqmc_time:.2f} \n')
        #     f.write('=' * width + '\n')
        # jax.clear_caches()
        # gc.collect()

    las_max = las_size.max()
    e_mp2 = np.sum(lno_emp2)
    e_ccsd = np.sum(lno_ecc)
    # e_afqmc = np.sum(lno_eqmc)
    # e_afqmc_err = np.sqrt(np.sum(lno_eqmc_err**2))
    # tot_ccsd_time = np.sum(ccsd_time)
    # tot_qmc_time = np.sum(qmc_time)

    # with open(f'lno_result.out', 'w') as f:
    #     width = 110
    #     f.write('=' * width + '\n')
    #     f.write(f'{"LNO-AFQMC Results":^{width}}\n')
    #     f.write('=' * width + '\n')

    #     f.write(f'{"Frag":>4s}  {"LAS Center":>14s}  {"LAS_SIZE":>8s}  '
    #             f'{"E(MP2)":>10s}  {"E(CCSD)":>10s}  '
    #             f'{"E(AFQMC)":>10s}  {"Error":>8s}  '
    #             f'{"t(CCSD)":>8s}  {"t(AFQMC)":>8s}\n')
    #     f.write('-' * width + '\n')
        
    #     for n, i in enumerate(run_frg_list):
    #         f.write(f"{i+1:4d}  {las_center[n]:>14s}  {las_size[n]:8d}  "
    #                 f"{lno_emp2[n]:10.8f}  {lno_ecc[n]:10.8f}  "
    #                 f"{lno_eqmc[n]:10.6f}  {lno_eqmc_err[n]:8.6f}  "
    #                 f"{ccsd_time[n]:8.2f}  {qmc_time[n]:8.2f}\n")
        
    #     f.write('-' * width + '\n')

    #     f.write(f'{"Sum":>4s}  {"":>14s}  {"":>8s}  '
    #             f'{e_mp2:10.8f}  {e_ccsd:10.8f}  '
    #             f'{e_afqmc:10.6f}  {e_afqmc_err:8.6f}  '
    #             f'{tot_ccsd_time:8.2f}  {tot_qmc_time:8.2f}\n')
    #     f.write('=' * width + '\n\n')

    #     f.write(f'LNO Threshold:          ({lno_thresh[0]:.2e}, {lno_thresh[1]:.2e})\n')
    #     f.write(f'MAX. Orbitals:          {las_max}\n')

    return e_mp2, e_ccsd

In [26]:
e_mp2, e_ccsd = run_afqmc(
        mf,
        lo_coeff = lo_coeff, 
        lo_coeff_file = 'lo_coeff.npz',
        frag_lolist = frag_lolist,
        nfrozen = mycc.frozen,
        thresh = 3e-5,
        qmc_options = {}, 
        chol_cut = 1e-5, 
        target_sto_error = 1e-3, 
        run_frg_list = None, 
        atom_group = None,
        )



=========================== RUNNING LNO-FRAGMENT 1/5 ===========================
LNO-Frgament Spin Type = mixed
LAS alpha: 5 occupied, 24 virtual
LAS beta:  5 occupied, 21 virtual
CCSD time: 0.316463 s
LNO-MP2 Orbital Energy: -0.16907233
LNO-CCSD Orbital Energy: -0.17672069


=========================== RUNNING LNO-FRAGMENT 2/5 ===========================
LNO-Frgament Spin Type = mixed
LAS alpha: 4 occupied, 16 virtual
LAS beta:  4 occupied, 16 virtual
CCSD time: 0.178193 s
LNO-MP2 Orbital Energy: -0.01641341
LNO-CCSD Orbital Energy: -0.01739036


=========================== RUNNING LNO-FRAGMENT 3/5 ===========================
LNO-Frgament Spin Type = mixed
LAS alpha: 5 occupied, 21 virtual
LAS beta:  5 occupied, 19 virtual
CCSD time: 0.224915 s
LNO-MP2 Orbital Energy: -0.01664219
LNO-CCSD Orbital Energy: -0.01772713


=========================== RUNNING LNO-FRAGMENT 4/5 ===========================
LNO-Frgament Spin Type = mixed
LAS alpha: 5 occupied, 15 virtual
LAS beta:  5 occupied

In [15]:
print(e_mp2, e_ccsd)

-0.4038544579142015 -0.4225857004942588


In [27]:
mf,
lo_coeff = lo_coeff
lo_coeff_file = 'lo_coeff.npz'
frag_lolist = frag_lolist
nfrozen = 2
thresh = 3e-5
qmc_options = {'n_eql': 3,
           'n_prop_steps': 50,
           'n_blocks': 100,
           'n_walkers': 100,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'uccsd_pt2',
           'dt':0.005,
           'use_gpu': True,
           }
chol_cut = 1e-5
target_sto_error = 1e-3
run_frg_list = [4]
atom_group = None
    
if lo_coeff is None:
    try:
        lo_coeff = np.load(lo_coeff_file)["lo_coeff"]
    except:
        raise ValueError(
            f"lo_coeff was not provided and could not be loaded from '{lo_coeff_file}'")

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag = len(frag_lolist)
if run_frg_list is None:
    run_frg_list = range(nfrag)

frag_lolist = [frag_lolist[i] for i in run_frg_list]
lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                       shape=(nfrag,), 
                       minval=0, 
                       maxval=100*nfrag
                       )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag
las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag, dtype='float64')
lno_ecc  = np.zeros(nfrag, dtype='float64')
lno_eqmc = np.zeros(nfrag, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag, dtype='float64')
ccsd_time = np.zeros(nfrag, dtype='float64')
qmc_time = np.zeros(nfrag, dtype='float64')

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" RUNNING {spin_type} LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
    print(msg.center(width, '='))
    if atom_group is not None:
        atom_msg = f"{atom_group[ifrag]}"
        print(f"Center Atom {atom_msg}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)
    # lno_coeff still connected to canonical mo_coeff unitarily

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')
    #     ao_message, ao_max = prep.ao_comp(mf, orbloc[spin_idx]) # TODO: make it general
    # else:
    #     ao_message, ao_max = prep.ao_comp(mf, orbloc)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = nactocc_a + nactocc_b
        nactvir = nactvir_a + nactvir_b
        print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS beta:  {nactocc_b} occupied, {nactvir_b} virtual')
    else:
        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        print(f'LAS occupied orbitals: {nactocc}')
        print(f'LAS virtual orbitals: {nactvir}')

    if spin_type == "unrestricted":
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f"CCSD time: {lnocc_time:.6f} s")
    print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    if atom_group:
        las_center[ifrag] = atom_msg
    # else:
    #     las_center[ifrag] = ao_max
    las_size[ifrag] = nactocc + nactvir
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time

    # project onto center lo space
    # <lno_actocc|orbloc> <orbloc|lno_actocc>
    if spin_type == "unrestricted":
        prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
                    uocc_loc[1] @ uocc_loc[1].T.conj()]
        qmc_options["trial"] = trial_base
        if 'ad' not in trial_base:
            if lno_elec_type == 'alpha':
                qmc_options["trial"] += '_alpha'
            elif lno_elec_type == 'beta':
                qmc_options["trial"] += '_beta'
    else:
        prjlo = uocc_loc @ uocc_loc.T.conj()

    qmc_options["seed"] = seeds[ifrag]
    prep.prep_afqmc_integral(
        mf,
        lno_coeff,
        t1,
        t2,
        lno_frozen,
        prjlo,
        qmc_options,
        chol_cut=chol_cut
        )



==================== RUNNING unrestricted LNO-FRAGMENT 5/5 =====================
LNO-Frgament Spin Type = mixed
LAS alpha: 5 occupied, 15 virtual
LAS beta:  2 occupied, 15 virtual
CCSD time: 0.240471 s
LNO-MP2 Orbital Energy: -0.01004497
LNO-CCSD Orbital Energy: -0.01220127
Calculating Effective Active Space One-electron Integrals
# Building JK matrix
build JK time: 0.005400 s
build ecore time: 0.005837 s
build h1eff time: 0.137994 s
Generating Cholesky Integrals
Constracting cLAS that span both Alpha and Beta active space
Naive cLAS Shape:  (43, 37)
Orthonormalize cLAS shape: (43, 37)
Smallest cLAS SVD Singular values: 0.0002789324766257659
cLAS projection torr: 1e-09
Minimum size of cLAS to span both Alpha and Beta LAS: 37
cLAS projection loss: (3.00e-14, 3.38e-14)
True Common LAS Shape:  (43, 37)
Find 42 AOs in cLAS with amplitude > 0.01
        AO Label     Amp
      2 H 2py       1.3246
     0 O 3dx2-y2    1.2625
      3 O 3py       1.2250
      4 H 2s        1.1864
      3 O 1s

In [28]:
from afqmc.lno_afqmc import lno_afqmc
lno_afqmc.run_lnoafqmc(options=qmc_options)

running AFQMC on GPU
Hostname: sharmagroup-rn
System Type: Linux
Machine Type: x86_64
Processor: x86_64
Using GPU.
System: Linux
Node Name: sharmagroup-rn
Release: 6.17.0-23-generic
Version: #23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2
Machine: x86_64
Processor: x86_64
AFQMC script: /home/sharmagroup/sharmagroup/afqmc_lab/afqmc/afqmc/lno_afqmc/ccsd_pt2/run_afqmc.py
---------------------------- AFQMC Sampling Started ----------------------------
Hostname: sharmagroup-rn
System Type: Linux
Machine Type: x86_64
Processor: x86_64
Using GPU.
System: Linux
Node Name: sharmagroup-rn
Release: 6.17.0-23-generic
Version: #23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2
Machine: x86_64
Processor: x86_64


E0512 14:41:47.226045   15279 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0512 14:41:47.234287   15212 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


Trial: uccsd_pt2(norb=(20, 17), nelec=(5, 2), n_batch=1, nchol_chunk=100, mix_precision=True)
Sampler: sampler_pt2(n_prop_steps=50, n_blocks=100, n_chol=194)
norb: (20, 17)
nelec: (5, 2)
nchol: 194
n_eql: 3
n_prop_steps: 50
n_blocks: 100
n_walkers: 100
n_batch: 1
seed: 489
walker_type: uhf
trial: uccsd_pt2
dt: 0.005
use_gpu: True
max_error: 0.0004472135954999579
mix_precision: True
nchol_chunk: 100
n_exp_terms: 6
Propagating with 100 walkers
Initial Orbital energy: -0.012201
-------------------------------- Equilibration ---------------------------------
inv_T        energy   runTime
 0.00   -151.156374      6.28
 0.50   -151.230703      7.91
 0.75   -151.266119      9.25
 1.00   -151.280882      9.31
------------------------------- Sampling Blocks --------------------------------
Target Final Error ~ 0.000447
   N      E(Guide)     Error      E(Orb)     Error  Kill_WK      Time
  10   -151.280230  0.003777   -0.012355  0.000051        0     15.67
  20   -151.284720  0.002854   -0.0123